## Notebook Overview

This notebook cleans and analyzes an electric-vehicle energy-consumption dataset. The workflow is connected: load data, inspect quality, create a clean copy, handle missing values, remove duplicates, investigate outliers, validate ranges, verify the result, compare datasets, save the cleaned file, and summarize energy consumption.

All existing code cells are preserved exactly. These Markdown cells explain each step and interpret the existing results.

In [21]:
import pandas as pd
import numpy as np

## Load the Dirty Dataset

The CSV is loaded into **df_dirty**, which remains the original baseline for auditing and comparison.

In [ ]:
df_dirty= pd.read_csv("../Data/EV_Energy_Consumption_Dirty.csv")

In [17]:
print("Rows:", df_dirty.shape[0])
print("Columns:", df_dirty.shape[1])

Rows: 5025
Columns: 19


## Step 1: Analyze Data Quality

The report counts missing values and exact duplicates. Missing values appear in **Speed_kmh**, **Battery_State_%**, **Temperature_C**, **Tire_Pressure_psi**, and **Energy_Consumption_kWh**. It also identifies **24 duplicate rows**, defining the problems addressed next.

In [18]:
# STEP 1: ANALYZE MISSING VALUES
# Count missing values for every column in the dirty dataset.

missing_report = pd.DataFrame({
    "Missing_Count": df_dirty.isnull().sum(),
    "Duplicte_Count": df_dirty.duplicated().sum(),
})

print("\nMissing Values Report:")
display(missing_report)
print(df_dirty.shape)


Missing Values Report:


,Missing_Count,Duplicte_Count
Vehicle_ID,0,24
Timestamp,0,24
Speed_kmh,1025,24
Acceleration_ms2,0,24
Battery_State_%,1005,24
Battery_Voltage_V,0,24
Battery_Temperature_C,0,24
Driving_Mode,0,24
Road_Type,0,24
Traffic_Condition,0,24


(5025, 19)


In [19]:
# Create a copy for the cleaning process.
# We do not clean df_dirty directly because we want to keep
# the dirty dataset available for comparison.

df_clean = df_dirty.copy()

## Step 1A: Handle Missing Target Values

The target **Energy_Consumption_kWh** cannot be safely imputed when missing, so those rows are removed. The output shows **1,001 rows removed** and **4,024 rows remaining**. Numerical feature gaps are then filled with medians, while the target remains excluded.

In [20]:
# 1A: HANDLE MISSING TARGET VALUES

# Energy_Consumption_kWh is our target variable.
# It is the value that the machine learning model will predict.
# We remove rows with a missing target because the correct
# target value is unknown for those records.

target_column = "Energy_Consumption_kWh"

if target_column in df_clean.columns:

    rows_before = len(df_clean)

    df_clean = df_clean.dropna(
        subset=[target_column]
    )

    rows_removed = rows_before - len(df_clean)

    print("\nRows removed due to missing target values:")
    print(rows_removed)
    print("Rows remaining:", len(df_clean))
    print()


Rows removed due to missing target values:
1001
Rows remaining: 4024



In [22]:
# ------------------------------------------------------------
# 1B: HANDLE MISSING NUMERICAL VALUES
# ------------------------------------------------------------

# We fill missing numerical feature values with their median.
# The median is the middle value and is less affected by
# extreme values than the mean.

numerical_columns = df_clean.select_dtypes(
    include=np.number
).columns

for column in numerical_columns:

    # Do not fill missing target values.
    if column == target_column:
        continue

    if df_clean[column].isnull().any():

        median_value = df_clean[column].median()

        df_clean[column] = df_clean[column].fillna(
            median_value
        )

        print(
            "Missing values filled in:",
            column,
            "| Method: Median"
        )

Missing values filled in: Speed_kmh | Method: Median
Missing values filled in: Battery_State_% | Method: Median
Missing values filled in: Temperature_C | Method: Median
Missing values filled in: Tire_Pressure_psi | Method: Median


## Step 2: Remove Duplicate Rows

There are **24 exact duplicate rows** before removal. After dropping them, the shape becomes **4,000 rows and 19 columns**, and the follow-up check reports **0 remaining duplicates**.

In [24]:
# STEP 2: FIND AND REMOVE DUPLICATES
# Count exact duplicate rows before removal.

duplicates_before = df_clean.duplicated().sum()

print("Duplicate Rows Before Cleaning:")
print(duplicates_before)

Duplicate Rows Before Cleaning:
24


In [25]:
# Remove exact duplicate rows.
# This avoids counting identical records more than once.

df_clean = df_clean.drop_duplicates().copy()
print(df_clean.shape)

(4000, 19)


In [26]:
# Count duplicates again to confirm the result.

duplicates_after = df_clean.duplicated().sum()

print("\nDuplicate Rows After Cleaning:")
print(duplicates_after)


Duplicate Rows After Cleaning:
0


## Step 3: Investigate Outliers

The IQR method calculates statistical boundaries for numerical columns. Values outside them are flagged for investigation, not automatically deleted, because an extreme vehicle or driving value may still be valid.

In [27]:
# STEP 3: INVESTIGATE OUTLIERS USING THE IQR METHOD

# IQR is a statistical method used to flag unusually low
# or high numerical values.
#
# Q1 = 25th percentile
# Q3 = 75th percentile
# IQR = Q3 - Q1
#
# Lower Bound = Q1 - 1.5 * IQR
# Upper Bound = Q3 + 1.5 * IQR
#
# Values outside these limits are flagged for investigation.
# They are not automatically considered errors.
print(" Outlier Investigation:")

numerical_columns = df_clean.select_dtypes(
    include=np.number
).columns

outlier_report = []

for column in numerical_columns:

    values = df_clean[column].dropna()

    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (df_clean[column] < lower_bound) |
        (df_clean[column] > upper_bound)
    )

    outlier_count = int(outlier_mask.sum())

    outlier_report.append({
        "Column": column,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": outlier_count
    })

outlier_report = pd.DataFrame(outlier_report)

display(outlier_report)
# Important decision:
# We do not delete every IQR outlier.
# A value may be extreme but still represent a real observation.
# We investigate values using the column meaning and valid ranges.

 Outlier Investigation:


,Column,Lower_Bound,Upper_Bound,Outlier_Count
0,Vehicle_ID,508.000000,2508.000000,0
1,Speed_kmh,-28.076255,146.562067,0
2,Acceleration_ms2,-5.928110,5.989128,0
3,Battery_State_%,1.669036,118.745533,1
4,Battery_Voltage_V,252.052921,450.641998,0
5,Battery_Temperature_C,-6.768568,61.585121,0
6,Driving_Mode,-2.000000,6.000000,0
7,Road_Type,-2.000000,6.000000,0
8,Traffic_Condition,-2.000000,6.000000,0
9,Slope_%,-12.603135,17.798777,0


## Step 4: Validate Clearly Invalid Values

Configured limits represent plausible ranges for selected vehicle, battery, weather, and tire variables. Values outside them are replaced with **NaN** and filled with the median of valid values. The limits should be confirmed against project documentation.

In [28]:
# STEP 4: CHECK CLEARLY INVALID VALUES

# These are example validity limits based on the meaning
# and units of selected columns.
# They should be checked against the dataset documentation.
# They are not universal rules for every electric vehicle.

valid_ranges = {
    "Speed_kmh": (0, 200),
    "Battery_State_%": (0, 100),
    "Battery_Voltage_V": (0, 600),
    "Temperature_C": (-50, 60),
    "Humidity_%": (0, 100),
    "Wind_Speed_ms": (0, 75),
    "Tire_Pressure_psi": (10, 60)
}

for column, limits in valid_ranges.items():

    if column not in df_clean.columns:
        continue

    minimum, maximum = limits

    # Identify values outside the configured range.
    invalid_mask = (
        (df_clean[column] < minimum) |
        (df_clean[column] > maximum)
    )

    invalid_count = int(invalid_mask.sum())

    print(
        f"\nInvalid values in {column}:",
        invalid_count
    )

    if invalid_count > 0:

        # Replace invalid feature values with NaN.
        # We then fill them using the median of the valid values.
        df_clean.loc[invalid_mask, column] = np.nan

        median_value = df_clean[column].median()

        if pd.notna(median_value):

            df_clean[column] = df_clean[column].fillna(
                median_value
            )


Invalid values in Speed_kmh: 0

Invalid values in Battery_State_%: 1

Invalid values in Battery_Voltage_V: 0

Invalid values in Temperature_C: 0

Invalid values in Humidity_%: 0

Invalid values in Wind_Speed_ms: 0

Invalid values in Tire_Pressure_psi: 1


## Validate the Target Variable

Negative energy consumption is removed as invalid; zero is retained because its meaning depends on the dataset definition. The output reports **0 invalid target values**, so no additional target rows are removed.

In [29]:
# Check the target separately.
# We remove clearly invalid negative energy consumption values.
# Zero is allowed here because its meaning should be verified
# against the dataset documentation and use case.

if target_column in df_clean.columns:

    invalid_target_mask = df_clean[target_column] < 0

    invalid_target_count = int(
        invalid_target_mask.sum()
    )

    print("\nInvalid target values:", invalid_target_count)

    df_clean = df_clean.loc[
        ~invalid_target_mask
    ].copy()


Invalid target values: 0


## Step 5: Verify the Cleaned Dataset

The missing-value check reports **0 remaining missing values**, and the duplicate check reports **0 remaining duplicate rows**. The final result contains **4,000 rows and 19 columns**, confirming that the main quality issues were addressed without changing the schema.

In [ ]:
# STEP 5: VERIFY THE CLEANED DATASET

# Check whether any missing values remain.

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sum())


Remaining Missing Values:
0


In [31]:

# Check whether any exact duplicate rows remain.

print("\nRemaining Duplicate Rows:")
print(df_clean.duplicated().sum())


Remaining Duplicate Rows:
0


In [33]:
# Display the final dataset size.

print("\nCleaned Dataset Shape:")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])


Cleaned Dataset Shape:
Rows: 4000
Columns: 19


## Inspect the Cleaned Data

The first five records provide a spot check that the expected vehicle, timestamp, driving, battery, environmental, distance, and energy fields remain available. The descriptive statistics show a count of **4,000** for numerical columns, a target mean near **8.46 kWh**, and a displayed maximum of **100 kWh** that should be reviewed against project context if unexpected.

In [35]:

# Display the first five cleaned records.

print("\nFirst Five Rows of Cleaned Dataset:")
display(df_clean.head().T.round(2))



First Five Rows of Cleaned Dataset:


,0,1,3,4,5
Vehicle_ID,1102,1435,1270,1106,1071
Timestamp,2024-01-01 00:00:00,2024-01-01 00:01:00,2024-01-01 00:03:00,2024-01-01 00:04:00,2024-01-01 00:05:00
Speed_kmh,111.507366,48.612323,38.579484,57.172438,27.12348
Acceleration_ms2,-2.773816,-0.796982,-2.111395,1.477883,-2.949872
Battery_State_%,60.786777,97.385534,60.786777,29.74016,60.786777
Battery_Voltage_V,378.091525,392.718377,358.128273,310.888162,345.912991
Battery_Temperature_C,25.314786,18.240755,28.980155,33.184551,19.758705
Driving_Mode,2,1,1,2,1
Road_Type,1,2,2,1,3
Traffic_Condition,1,1,2,1,1


In [36]:

# Display summary statistics after cleaning.

print("\nStatistics After Cleaning:")
display(df_clean.describe().T.round(2))


Statistics After Cleaning:


,count,mean,std,min,25%,50%,75%,max
Vehicle_ID,4000.0,1506.52,288.70,1000.00,1258.00,1510.00,1758.00,1999.00
Speed_kmh,4000.0,59.31,30.88,0.00,37.41,58.55,81.07,119.95
Acceleration_ms2,4000.0,0.01,1.73,-3.00,-1.46,-0.02,1.52,3.00
Battery_State_%,4000.0,60.18,20.58,20.02,45.57,60.79,74.81,99.99
Battery_Voltage_V,4000.0,351.16,29.11,300.03,326.52,351.56,376.17,399.99
Battery_Temperature_C,4000.0,27.44,10.04,10.00,18.86,27.48,35.95,44.99
Driving_Mode,4000.0,2.01,0.81,1.00,1.00,2.00,3.00,3.00
Road_Type,4000.0,2.00,0.81,1.00,1.00,2.00,3.00,3.00
Traffic_Condition,4000.0,2.01,0.81,1.00,1.00,2.00,3.00,3.00
Slope_%,4000.0,2.52,4.37,-4.99,-1.20,2.48,6.40,10.00


## Step 6: Compare Before and After Cleaning

The comparison measures the effect of cleaning. The cleaned data has fewer rows, keeps **19 columns**, and should have zero missing cells and duplicate rows. The dirty and cleaned numerical summaries show how imputation and filtering changed the distributions.

In [37]:
# STEP 6: COMPARE THE DIRTY AND CLEANED DATASETS

# This comparison shows the effect of our cleaning decisions.

comparison = pd.DataFrame({
    "Dirty Dataset": [
        df_dirty.shape[0],
        df_dirty.shape[1],
        int(df_dirty.isnull().sum().sum()),
        int(df_dirty.duplicated().sum())
    ],
    "Cleaned Dataset": [
        df_clean.shape[0],
        df_clean.shape[1],
        int(df_clean.isnull().sum().sum()),
        int(df_clean.duplicated().sum())
    ]
}, index=[
    "Number of Rows",
    "Number of Columns",
    "Missing Cells",
    "Duplicate Rows"
])

print("\nBefore and After Cleaning:")
display(comparison)

# Compare numerical statistics before and after cleaning.
common_numerical_columns = [
    column
    for column in df_dirty.select_dtypes(include=np.number).columns
    if column in df_clean.columns
]



Before and After Cleaning:


,Dirty Dataset,Cleaned Dataset
Number of Rows,5025,4000
Number of Columns,19,19
Missing Cells,5038,0
Duplicate Rows,24,0


In [39]:

print("Dirty Dataset Statistics:")
display(df_dirty[common_numerical_columns].describe().T)

Dirty Dataset Statistics:


,count,mean,std,min,25%,50%,75%,max
Vehicle_ID,5025.0,1505.935522,290.747185,1000.000000,1253.000000,1510.000000,1761.000000,1999.000000
Speed_kmh,4000.0,59.613821,35.239092,0.003686,29.584775,58.845695,89.342317,500.000000
Acceleration_ms2,5025.0,-0.020334,1.724236,-2.999683,-1.489269,-0.046703,1.473019,2.998039
Battery_State_%,4020.0,60.371785,23.248081,20.020196,40.043652,60.957265,80.629532,150.000000
Battery_Voltage_V,5025.0,351.267673,30.367620,300.004812,326.646694,351.537433,376.004721,1000.000000
Battery_Temperature_C,5025.0,27.505227,10.017572,10.003858,18.889561,27.563704,35.911497,44.985943
Driving_Mode,5025.0,2.010945,0.810264,1.000000,1.000000,2.000000,3.000000,3.000000
Road_Type,5025.0,1.997612,0.808490,1.000000,1.000000,2.000000,3.000000,3.000000
Traffic_Condition,5025.0,2.010348,0.812358,1.000000,1.000000,2.000000,3.000000,3.000000
Slope_%,5025.0,2.508856,4.362668,-4.991079,-1.233963,2.467071,6.379088,9.998358


In [40]:
print("Cleaned Dataset Statistics:")
display(df_clean[common_numerical_columns].describe().T)

Cleaned Dataset Statistics:


,count,mean,std,min,25%,50%,75%,max
Vehicle_ID,4000.0,1506.519500,288.695268,1000.000000,1258.000000,1510.000000,1758.000000,1999.000000
Speed_kmh,4000.0,59.306859,30.880357,0.003686,37.413116,58.548023,81.072697,119.946924
Acceleration_ms2,4000.0,0.010618,1.728006,-2.999683,-1.459146,-0.022814,1.520164,2.998039
Battery_State_%,4000.0,60.180424,20.575485,20.020196,45.572722,60.786777,74.809929,99.993986
Battery_Voltage_V,4000.0,351.162777,29.106100,300.030708,326.523825,351.558089,376.171094,399.990098
Battery_Temperature_C,4000.0,27.440735,10.042238,10.003858,18.864065,27.477515,35.952487,44.985943
Driving_Mode,4000.0,2.008500,0.809377,1.000000,1.000000,2.000000,3.000000,3.000000
Road_Type,4000.0,2.000750,0.809885,1.000000,1.000000,2.000000,3.000000,3.000000
Traffic_Condition,4000.0,2.014250,0.813764,1.000000,1.000000,2.000000,3.000000,3.000000
Slope_%,4000.0,2.522834,4.373583,-4.991079,-1.202418,2.484552,6.398060,9.998358


## Step 7: Save the Cleaned Dataset

The final **df_clean** table is exported to a separate CSV. The dirty file is not overwritten, preserving both versions for reproducibility.

In [44]:
# ============================================================
# STEP 7: SAVE THE CLEANED DATASET
# ============================================================

# Save the final cleaned data to a new CSV file.
# The original and dirty files remain separate.

df_clean.to_csv(
    r"C:\Users\njnay\Desktop\EVEnergyPrediction\Data\EV_Energy_Consumption_Cleaned.csv",
    index=False
)

print("Cleaning Process Completed Successfully")
print("Dirty file: EV_Energy_Consumption_Dirty.csv")
print("Cleaned file: EV_Energy_Consumption_Cleaned.csv")

Cleaning Process Completed Successfully
Dirty file: EV_Energy_Consumption_Dirty.csv
Cleaned file: EV_Energy_Consumption_Cleaned.csv


## Analyze Average Energy by Driving Mode

This grouping calculates mean energy consumption by driving mode and sorts the results from highest to lowest, showing which mode uses more energy on average.

In [45]:
# Grouping & Sorting:
df_clean.groupby('Driving_Mode')['Energy_Consumption_kWh'].mean().sort_values(ascending=False)

Driving_Mode
3    8.523408
2    8.457573
1    8.382610
Name: Energy_Consumption_kWh, dtype: float64

## Analyze Average Energy by Road Type

This grouping calculates mean energy consumption by road type and sorts the results from lowest to highest, comparing road categories before descriptive labels are added.

In [46]:
df_clean.groupby('Road_Type')['Energy_Consumption_kWh'].mean().sort_values(ascending=True)

Road_Type
1    8.353239
2    8.487968
3    8.522507
Name: Energy_Consumption_kWh, dtype: float64

## Add Descriptive Road-Type Names

The lookup maps codes to labels: **1 = Highway**, **2 = Urban**, and **3 = Rural**. A left merge keeps every cleaned observation and adds its readable road-type label.

In [47]:
#mergging
mode_name = pd.DataFrame({
    'Road_Type': [1, 2, 3],
    'Mode': ['Highway', 'Urban', 'Rural']
})

df_clean['Road_Type'] = df_clean['Road_Type'].astype(int)
mode_name['Road_Type'] = mode_name['Road_Type'].astype(int)

df = pd.merge(df_clean, mode_name, on='Road_Type', how='left')
df[['Road_Type', 'Mode', 'Energy_Consumption_kWh']].tail()

,Road_Type,Mode,Energy_Consumption_kWh
3995,1,Highway,10.089577
3996,2,Urban,9.600692
3997,3,Rural,9.148392
3998,1,Highway,5.581431
3999,1,Highway,7.841474


## Compare Driving Mode and Road Type Together

The pivot table calculates mean energy consumption for each driving-mode and road-type combination, connecting the previous analyses and showing how road context changes energy use.

In [48]:
#pivot table
pivot_energy = df.pivot_table(
    index='Driving_Mode',
    columns='Road_Type',
    values='Energy_Consumption_kWh',
    aggfunc='mean'
)
print("Average Energy Consumption by Driving Mode & Road Type:")
pivot_energy

Average Energy Consumption by Driving Mode & Road Type:


Road_Type,1,2,3
Driving_Mode,,,
1,8.251043,8.454473,8.429902
2,8.420096,8.497651,8.456844
3,8.375808,8.513608,8.670256
